In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

porto_seguro_safe_driver_prediction_path = kagglehub.competition_download('porto-seguro-safe-driver-prediction')

print('Data source import complete.')


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data_path = '/kaggle/input/porto-seguro-safe-driver-prediction/'

In [ ]:
train = pd.read_csv(data_path + 'train.csv', index_col='id')
test = pd.read_csv(data_path + 'test.csv', index_col='id')
submission = pd.read_csv(data_path + 'sample_submission.csv', index_col='id')

# 피처 엔지니어링

In [ ]:
all_data = pd.concat([train, test], ignore_index=True)
all_data = all_data.drop('target', axis=1)

In [ ]:
all_features = all_data.columns
all_data

## 명목형 피처 원-핫 인코딩

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
cat_features = [feature for feature in all_features if 'cat' in feature]

In [ ]:
onehot_encoder = OneHotEncoder()
encoded_cat_matrix = onehot_encoder.fit_transform(all_data[cat_features])
encoded_cat_matrix

## 필요 없는 피처 제거

In [ ]:
drop_features = ['ps_ind_14', 'ps_ind_10_bin', 'ps_ind_11_bin',
                 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_car_14']

In [ ]:
remaining_features = [feature for feature in all_features
                      if ('cat' not in feature and
                          'calc' not in drop_features)]

In [ ]:
from scipy import sparse

In [ ]:
all_data_sprs = sparse.hstack([sparse.csr_matrix(all_data[remaining_features]),
                               encoded_cat_matrix],
                              format='csr')

## 데이터 나누기

In [ ]:
num_train = len(train)

In [ ]:
X = all_data_sprs[:num_train]
X_test = all_data_sprs[num_train:]

In [ ]:
y = train['target'].values

# 평가지표 계산 함수 작성

## 정규화 지니계수 계산 함수

In [ ]:
def eval_gini(y_true, y_pred):
    # 실제값과 예측값의 크기가 서로 같은지 확인 (값이 다르면 오류 발생)
    assert y_true.shape == y_pred.shape

    n_samples = y_true.shape[0]
    L_mid = np.linspace(1/n_samples, 1, n_samples)

    # 1) 예측값에 대한 지니계수
    pred_order = y_true[y_pred.argsort()]
    L_pred = np.cumsum(pred_order) / np.sum(pred_order)
    G_pred = np.sum(L_mid - L_pred)

    # 2) 예측이 완벽할 때 지니계수
    true_order = y_true[y_true.argsort()]
    L_true = np.cumsum(true_order) / np.sum(true_order)
    G_true = np.sum(L_mid - L_true)

    # 정규화된 지니계수
    return G_pred / G_true

In [ ]:
# 모델 훈련 시 검증 파라미터에 전달하기 위한 함수
def gini(preds, dtrain):
    labels = dtrain.get_label()
    return 'gini', eval_gini(labels, preds), True
        # 평가지표 이름, 평가 점수, 평가 점수가 높을수록 좋은지 여부

# 모델 훈련 및 성능 검증

## OOF 방식으로 LightGBM 훈련

### OOF 검증 방식

In [ ]:
from sklearn.model_selection import StratifiedKFold

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=1991)

In [ ]:
params = {'objective': 'binary',
          'learning_rate': 0.1,
          'force_row_wise': True,
          'random_state': 0}

In [ ]:
# OOF 방식으로 훈련된 모델로 검증 데이터 타깃값을 예측한 확률을 담을 1차원 배열
oof_val_preds = np.zeros(X.shape[0])

# OOF 방식으로 훈련된 모델로 테스트 데이터 타깃값을 예측한 확률을 담을 1차원 배열
oof_test_preds = np.zeros(X_test.shape[0])

### LightGBM 모델 훈련

In [ ]:
import lightgbm as lgb

In [ ]:
# OOF 방식으로 모델 훈련, 검증, 예측
for idx, (train_dix, valid_idx) in enumerate(folds.split(X, y)):
    print('#'*40, f'폴드 {idx+1} / 폴드 {folds.n_splits}', '#'*40)

    # 훈련용 데이터, 검증용 데이터 설정
    X_train, y_train = X[train_dix], y[train_dix]
    X_valid, y_valid = X[valid_idx], y[valid_idx]

    # LightBGM 전용 데이터셋 생성
    dtrain = lgb.Dataset(X_train, y_train)
    dvalid = lgb.Dataset(X_valid, y_valid)

    # LightBGM 모델 훈련
    lgb_model = lgb.train(params=params,            # 훈령용 하이퍼파라미터
                          train_set=dtrain,         # 훈련 데이터셋
                          num_boost_round=1000,     # 부스팅 반복 횟수
                          valid_sets=dvalid,        # 성능 평가용 검증 데이터셋
                          feval=gini,               # 검증용 평가지표
                          # early_stopping_rounds=100,# 조기종료 조건
                          # verbose_eval=100,         # 100번째마다 점수 출력
                          callbacks=[
                              lgb.early_stopping(stopping_rounds=3),
                              lgb.log_evaluation(100)
                        ])

    # 테스트 데이터를 활용해 OOF 예측
    oof_test_preds += lgb_model.predict(X_test)/folds.n_splits
    # 모델 성능 평가를 위한 검증 데이터 타깃값 예측
    oof_val_preds[valid_idx] += lgb_model.predict(X_valid)

    # 검증 데이터 예측 확률에 대한 정규화 지니계수
    gini_score = eval_gini(y_valid, oof_val_preds[valid_idx])
    print(f'폴드 {idx+1} 지니계수: {gini_score}/n')

In [ ]:
print('OOF 검증 데이터 지니계수: ', eval_gini(y, oof_val_preds))

# 예측 및 결과 제출

In [ ]:
submission['target'] = oof_test_preds
submission.to_csv('submission.csv')